In [ ]:
```xml
<VSCode.Cell language="markdown">
# 📊 Distributed Tracing & Observability 2.0 Guide

**Building comprehensive observability with distributed tracing, correlation, and unified telemetry**

This guide covers:
- Distributed tracing concepts
- Trace context propagation
- Span instrumentation
- Correlation IDs
- Sampling strategies
- Trace aggregation
- OpenTelemetry framework
- Observability as code
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Distributed Tracing Fundamentals

### Trace Context & Spans
```python
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from datetime import datetime
import uuid
import json

@dataclass
class TraceContext:
    """Distributed trace context"""
    trace_id: str
    span_id: str
    parent_span_id: Optional[str]
    trace_flags: str = "01"  # Sampled
    trace_state: Dict = field(default_factory=dict)

@dataclass
class Span:
    """Individual span in trace"""
    trace_id: str
    span_id: str
    parent_span_id: Optional[str]
    span_name: str
    start_time: datetime
    end_time: Optional[datetime]
    duration_ms: float = 0.0
    status: str = "unset"  # unset, ok, error
    attributes: Dict = field(default_factory=dict)
    events: List[Dict] = field(default_factory=list)
    error_message: Optional[str] = None

class TraceContextManager:
    """Manage trace context propagation"""
    
    def __init__(self):
        self.active_traces: Dict[str, TraceContext] = {}
        self.trace_stack: List[TraceContext] = []
    
    def create_trace(self) -> TraceContext:
        """Create new trace"""
        
        trace_id = uuid.uuid4().hex
        span_id = uuid.uuid4().hex[:16]
        
        context = TraceContext(
            trace_id=trace_id,
            span_id=span_id,
            parent_span_id=None
        )
        
        self.active_traces[trace_id] = context
        self.trace_stack.append(context)
        
        return context
    
    def create_child_span(self, trace_context: TraceContext) -> TraceContext:
        """Create child span"""
        
        child_span_id = uuid.uuid4().hex[:16]
        
        child_context = TraceContext(
            trace_id=trace_context.trace_id,
            span_id=child_span_id,
            parent_span_id=trace_context.span_id
        )
        
        return child_context
    
    def extract_context_from_headers(self, headers: Dict) -> Optional[TraceContext]:
        """Extract trace context from HTTP headers (W3C format)"""
        
        traceparent = headers.get('traceparent', '')
        
        if not traceparent:
            return None
        
        try:
            parts = traceparent.split('-')
            
            if len(parts) < 3:
                return None
            
            trace_id = parts[1]
            parent_span_id = parts[2]
            
            context = TraceContext(
                trace_id=trace_id,
                span_id=uuid.uuid4().hex[:16],
                parent_span_id=parent_span_id
            )
            
            return context
        
        except Exception as e:
            print(f"Failed to extract trace context: {e}")
            return None
    
    def inject_context_to_headers(self, context: TraceContext) -> Dict:
        """Inject trace context into HTTP headers"""
        
        traceparent = f"00-{context.trace_id}-{context.span_id}-{context.trace_flags}"
        
        return {
            'traceparent': traceparent,
            'tracestate': json.dumps(context.trace_state) if context.trace_state else ''
        }

class SpanRecorder:
    """Record spans"""
    
    def __init__(self):
        self.spans: List[Span] = []
        self.span_index: Dict[str, Span] = {}
    
    def start_span(self, context: TraceContext, span_name: str) -> Span:
        """Start span"""
        
        span = Span(
            trace_id=context.trace_id,
            span_id=context.span_id,
            parent_span_id=context.parent_span_id,
            span_name=span_name,
            start_time=datetime.utcnow(),
            end_time=None
        )
        
        return span
    
    def end_span(self, span: Span, status: str = "ok", error: Optional[str] = None):
        """End span"""
        
        span.end_time = datetime.utcnow()
        span.duration_ms = (span.end_time - span.start_time).total_seconds() * 1000
        span.status = status
        span.error_message = error
        
        self.spans.append(span)
        self.span_index[span.span_id] = span
    
    def record_event(self, span: Span, event_name: str, attributes: Dict = None):
        """Record event on span"""
        
        span.events.append({
            'name': event_name,
            'timestamp': datetime.utcnow().isoformat(),
            'attributes': attributes or {}
        })
    
    def get_trace(self, trace_id: str) -> List[Span]:
        """Get all spans for trace"""
        
        return [s for s in self.spans if s.trace_id == trace_id]
```

### Context Propagation
```python
class ContextPropagator:
    """Propagate trace context"""
    
    def __init__(self):
        self.propagators: Dict[str, callable] = {}
    
    def register_propagator(self, format_name: str, extractor: callable, injector: callable):
        """Register propagator"""
        
        self.propagators[format_name] = {
            'extract': extractor,
            'inject': injector
        }
    
    def extract(self, format_name: str, carrier: Dict) -> Optional[TraceContext]:
        """Extract context"""
        
        if format_name not in self.propagators:
            return None
        
        extractor = self.propagators[format_name]['extract']
        return extractor(carrier)
    
    def inject(self, format_name: str, context: TraceContext, carrier: Dict) -> Dict:
        """Inject context"""
        
        if format_name not in self.propagators:
            return carrier
        
        injector = self.propagators[format_name]['inject']
        return injector(context, carrier)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Sampling & Aggregation

### Sampling Strategies
```python
import random

class SamplingStrategy:
    """Base sampling strategy"""
    
    def should_sample(self, trace_id: str, attributes: Dict = None) -> bool:
        raise NotImplementedError

class FixedRateSampler(SamplingStrategy):
    """Sample fixed percentage of traces"""
    
    def __init__(self, sample_rate: float = 0.1):
        self.sample_rate = max(0.0, min(1.0, sample_rate))
    
    def should_sample(self, trace_id: str, attributes: Dict = None) -> bool:
        """Sample based on fixed rate"""
        return random.random() < self.sample_rate

class AdaptiveSampler(SamplingStrategy):
    """Adapt sampling based on load"""
    
    def __init__(self, initial_rate: float = 0.1):
        self.current_rate = initial_rate
        self.min_rate = 0.01
        self.max_rate = 1.0
        self.metrics_window: List[Dict] = []
    
    def adjust_for_load(self, error_rate: float, latency_p99_ms: float):
        """Adjust sampling rate"""
        
        # Reduce sampling if high error rate
        if error_rate > 0.05:
            self.current_rate = max(self.min_rate, self.current_rate * 0.8)
        
        # Reduce sampling if high latency
        if latency_p99_ms > 1000:
            self.current_rate = max(self.min_rate, self.current_rate * 0.9)
        
        # Increase sampling if healthy
        if error_rate < 0.01 and latency_p99_ms < 100:
            self.current_rate = min(self.max_rate, self.current_rate * 1.1)
    
    def should_sample(self, trace_id: str, attributes: Dict = None) -> bool:
        """Sample with adaptive rate"""
        return random.random() < self.current_rate

class PrioritySampler(SamplingStrategy):
    """Sample based on priority"""
    
    def __init__(self, default_rate: float = 0.1):
        self.default_rate = default_rate
        self.priority_rates: Dict[str, float] = {
            'critical': 1.0,
            'high': 0.5,
            'normal': 0.1,
            'low': 0.01
        }
    
    def should_sample(self, trace_id: str, attributes: Dict = None) -> bool:
        """Sample based on priority"""
        
        if not attributes:
            return random.random() < self.default_rate
        
        priority = attributes.get('priority', 'normal')
        rate = self.priority_rates.get(priority, self.default_rate)
        
        return random.random() < rate
```

### Trace Aggregation
```python
class TraceAggregator:
    """Aggregate traces"""
    
    def __init__(self):
        self.traces: Dict[str, Dict] = {}
        self.service_stats: Dict[str, Dict] = {}
    
    def ingest_trace(self, trace_id: str, spans: List[Span]):
        """Ingest trace"""
        
        trace_data = {
            'trace_id': trace_id,
            'spans': spans,
            'duration_ms': self._calculate_trace_duration(spans),
            'service_count': len(set(s.attributes.get('service', '') for s in spans)),
            'error': any(s.status == 'error' for s in spans),
            'ingested_at': datetime.utcnow().isoformat()
        }
        
        self.traces[trace_id] = trace_data
        
        # Update service stats
        self._update_service_stats(spans)
    
    def _calculate_trace_duration(self, spans: List[Span]) -> float:
        """Calculate trace duration"""
        
        if not spans:
            return 0.0
        
        start_time = min(s.start_time for s in spans)
        end_time = max(s.end_time for s in spans if s.end_time)
        
        return (end_time - start_time).total_seconds() * 1000
    
    def _update_service_stats(self, spans: List[Span]):
        """Update statistics per service"""
        
        for span in spans:
            service = span.attributes.get('service', 'unknown')
            
            if service not in self.service_stats:
                self.service_stats[service] = {
                    'span_count': 0,
                    'error_count': 0,
                    'total_duration_ms': 0.0
                }
            
            self.service_stats[service]['span_count'] += 1
            
            if span.status == 'error':
                self.service_stats[service]['error_count'] += 1
            
            self.service_stats[service]['total_duration_ms'] += span.duration_ms
    
    def query_traces(self, service: str = None, status: str = None) -> List[Dict]:
        """Query traces"""
        
        results = []
        
        for trace_id, trace_data in self.traces.items():
            # Filter by service
            if service:
                if service not in [s.attributes.get('service', '') for s in trace_data['spans']]:
                    continue
            
            # Filter by status
            if status:
                trace_has_error = any(s.status == 'error' for s in trace_data['spans'])
                if status == 'error' and not trace_has_error:
                    continue
                if status == 'ok' and trace_has_error:
                    continue
            
            results.append(trace_data)
        
        return results
    
    def get_service_metrics(self, service: str) -> Optional[Dict]:
        """Get metrics for service"""
        
        if service not in self.service_stats:
            return None
        
        stats = self.service_stats[service]
        
        return {
            'service': service,
            'span_count': stats['span_count'],
            'error_count': stats['error_count'],
            'error_rate': (stats['error_count'] / stats['span_count']) if stats['span_count'] > 0 else 0,
            'avg_duration_ms': stats['total_duration_ms'] / stats['span_count'] if stats['span_count'] > 0 else 0
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. OpenTelemetry Integration

### Instrumenting Services
```python
class InstrumentationContext:
    """Context for instrumentation"""
    
    def __init__(self, service_name: str, version: str):
        self.service_name = service_name
        self.version = version
        self.trace_provider = SpanRecorder()
        self.context_manager = TraceContextManager()
    
    def create_tracer(self, name: str) -> 'Tracer':
        """Create tracer"""
        return Tracer(name, self)

class Tracer:
    """Instrument code with spans"""
    
    def __init__(self, name: str, context: InstrumentationContext):
        self.name = name
        self.context = context
        self.current_trace: Optional[TraceContext] = None
    
    def start_trace(self) -> 'TracingContext':
        """Start trace"""
        
        self.current_trace = self.context.context_manager.create_trace()
        return TracingContext(self, self.current_trace)
    
    def start_span(self, span_name: str) -> 'SpanContext':
        """Start span"""
        
        if not self.current_trace:
            self.start_trace()
        
        span = self.context.trace_provider.start_span(self.current_trace, span_name)
        return SpanContext(self, span)

class TracingContext:
    """Context manager for tracing"""
    
    def __init__(self, tracer: 'Tracer', trace_context: TraceContext):
        self.tracer = tracer
        self.trace_context = trace_context
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type:
            print(f"Trace ended with error: {exc_type}")

class SpanContext:
    """Context manager for spans"""
    
    def __init__(self, tracer: 'Tracer', span: Span):
        self.tracer = tracer
        self.span = span
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type:
            self.span.status = 'error'
            self.span.error_message = str(exc_val)
        
        self.tracer.context.trace_provider.end_span(self.span)
    
    def set_attribute(self, key: str, value):
        """Set span attribute"""
        self.span.attributes[key] = value
    
    def add_event(self, event_name: str, attributes: Dict = None):
        """Add event to span"""
        self.tracer.context.trace_provider.record_event(self.span, event_name, attributes)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Distributed Tracing Checklist

✅ **Instrumentation**
- [ ] Trace context created
- [ ] Spans recorded
- [ ] Context propagation
- [ ] Headers injected/extracted
- [ ] Service names set

✅ **Sampling**
- [ ] Sampling strategy chosen
- [ ] Rate configured
- [ ] Adaptive sampling
- [ ] Priority sampling
- [ ] Performance optimized

✅ **Trace Aggregation**
- [ ] Traces stored
- [ ] Service metrics tracked
- [ ] Query functionality
- [ ] Retention policy
- [ ] Archival strategy

✅ **Observability**
- [ ] Trace visualization
- [ ] Error tracking
- [ ] Performance analysis
- [ ] Service map
- [ ] Latency percentiles

✅ **Operational**
- [ ] Documentation
- [ ] Team training
- [ ] Monitoring dashboard
- [ ] Alert thresholds
- [ ] Runbooks
</VSCode.Cell>
```